In [2]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate_hkqai").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"new_dataset/{data_set}.json") as f:
    json_data = json.load(f)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict_ccpvdz"]
    # full_subset_dict = json.load(f)["full_subset_dict_test"]

data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for data_path in data_path_list:
        data = pd.read_csv(data_path)
        data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
        data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
        data_dft = data["dft_ene"].to_numpy() * 627.5094733748099
        data_scf = data["scf_ene"].to_numpy() * 627.5094733748099
        data_cc = data["cc_ene"].to_numpy() * 627.5094733748099

        data_error_scf_ele = data["error_scf_ele"].to_numpy()
        data_error_dft_ele = data["error_dft_ele"].to_numpy()
        data_error_scf_dip = data["error_scf_dip"].to_numpy()
        data_error_dft_dip = data["error_dft_dip"].to_numpy()
        data_subset[f"{data_path_name}_summary"] = {
            "error_scf_ele": data_error_scf_ele,
            "error_dft_ele": data_error_dft_ele,
            "error_scf_dip": data_error_scf_dip,
            "error_dft_dip": data_error_dft_dip,
        }

        if "delta_d3bj" in data.columns:
            data_d3bj = data["delta_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_d3bj = np.zeros_like(data_dft)
        if "delta_d3zero" in data.columns:
            data_d3zero = data["delta_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_d3zero = np.zeros_like(data_dft)

        if "modified_dft_d3bj" in data.columns:
            data_dft_d3bj = data["modified_dft_d3bj"].to_numpy() * 627.5094733748099
            # data_dft_d3bj = data_d3bj
        else:
            data_dft_d3bj = data_d3bj
        if "modified_dft_d3zero" in data.columns:
            data_dft_d3zero = data["modified_dft_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_dft_d3zero = data_d3zero

        if "modified_ai_d3bj" in data.columns:
            data_ai_d3bj = data["modified_ai_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3bj = data_d3bj
        if "modified_ai_d3zero" in data.columns:
            data_ai_d3zero = data["modified_ai_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3zero = data_d3zero

        del data_d3bj, data_d3zero

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "dft_d3bj": [],
                "dft_d3zero": [],
                "ai": [],
                "ai_d3bj": [],
                "ai_d3zero": [],
                "cc": [],
                "error_scf_ele": [],
                "error_dft_ele": [],
                "error_scf_dip": [],
                "error_dft_dip": [],
            }

            if i_subset == "BH76RC":
                molecular_list = json_data["molecule_BH76"]
            else:
                molecular_list = json_data[f"molecule_{i_subset}"]

            for i_molecule_name in molecular_list:
                col = np.where(data_name == i_molecule_name)[0]
                if col.size != 1:
                    if verbose > 0:
                        print(
                            f"Warning: {i_molecule_name} not found in {data_path.stem} data file"
                        )
                    continue
                data_subset[name_subset]["error_scf_ele"].append(
                    data_error_scf_ele[col[0]]
                )
                data_subset[name_subset]["error_dft_ele"].append(
                    data_error_dft_ele[col[0]]
                )
                data_subset[name_subset]["error_scf_dip"].append(
                    data_error_scf_dip[col[0]]
                )
                data_subset[name_subset]["error_dft_dip"].append(
                    data_error_dft_dip[col[0]]
                )

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_dft_d3bj = 0
                atomic_energy_dft_d3zero = 0
                atomic_energy_ai = 0
                atomic_energy_ai_d3bj = 0
                atomic_energy_ai_d3zero = 0
                atomic_energy_cc = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                        atomic_energy_dft_d3bj += data_dft_d3bj[col[0]] * stoichiometry
                        atomic_energy_dft_d3zero += (
                            data_dft_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_ai += data_scf[col[0]] * stoichiometry
                        atomic_energy_ai_d3bj += data_ai_d3bj[col[0]] * stoichiometry
                        atomic_energy_ai_d3zero += (
                            data_ai_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_cc += data_cc[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["dft_d3bj"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3bj
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["dft_d3zero"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["ai"].append(
                        abs(atomic_energy_ai - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3bj"].append(
                        abs(atomic_energy_ai + atomic_energy_ai_d3bj - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3zero"].append(
                        abs(
                            atomic_energy_ai
                            + atomic_energy_ai_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

            if verbose > 1:
                argsort_atomic_energy_ai = np.argsort(data_subset[name_subset]["ai"])[
                    ::-1
                ][:5]
                argsort_atomic_energy_dft = np.argsort(data_subset[name_subset]["dft"])[
                    ::-1
                ][:5]
                for i in range(len(argsort_atomic_energy_ai)):
                    i_reaction_name = data_subset[name_subset]["name"][
                        argsort_atomic_energy_ai[i]
                    ]
                    i_reaction = json_data[f"reaction-{i_subset}"][i_reaction_name]
                    print(
                        f"Top {i+1} AI: {data_subset[name_subset]['ai'][argsort_atomic_energy_ai[i]]} kcal/mol, {i_reaction_name} in {name_subset}",
                    )
                    systems_list = i_reaction["systems"]
                    stoichiometry_list = i_reaction["stoichiometry"]
                    for j in range(len(systems_list)):
                        mole_name = (
                            systems_list[j]
                            if i_subset == "BH76RC"
                            else f"{i_subset}-{systems_list[j]}"
                        )
                        stoichiometry = int(stoichiometry_list[j])
                        if mole_name in json_data:
                            if isinstance(json_data[mole_name], str):
                                mole_name = json_data[mole_name]
                        print(f"  {stoichiometry} * {mole_name}", end="")
                        col = np.where(data_name == mole_name)[0]
                        if col.size == 1:
                            error_energy_ai = data_scf[col[0]] - data_cc[col[0]]
                            print(f"  {stoichiometry} * {error_energy_ai}", end="")
                    print()

                for i in range(len(argsort_atomic_energy_dft)):
                    print(
                        f"Top {i+1} DFT: {data_subset[name_subset]['dft'][argsort_atomic_energy_dft[i]]} kcal/mol"
                    )

data_path_name_list = [
    data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for data_path in data_path_list
]
header = pd.MultiIndex.from_product(
    [
        data_path_name_list,
        [
            "AI",
            "DFT",
            "AI_D3BJ",
            "DFT_D3BJ",
            "AI_D3ZERO",
            "DFT_D3ZERO",
            "Processed",
        ],
    ],
    names=["data_path", "Disp type"],
)

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)

df_summary_subset_ele = pd.DataFrame(
    columns=pd.MultiIndex.from_product(
        [
            data_path_name_list,
            [
                "error_scf_ele",
                "error_dft_ele",
                "error_scf_dip",
                "error_dft_dip",
            ],
        ],
        names=["data_path", "Ele type"],
    )
)

for data_path in data_path_list:
    mean_absolute_deviation_list = []
    data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_dip"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_dip"]
    )

    for name_set, subset_list_ in full_subset_dict.items():
        subset_ai = {}
        wtmad_1_ai = {}
        wtmad_2_ai = {}
        subset_dft = {}
        wtmad_1_dft = {}
        wtmad_2_dft = {}

        for d3_name in ["", "_d3bj", "_d3zero"]:
            subset_ai[d3_name] = []
            wtmad_1_ai[d3_name] = []
            wtmad_2_ai[d3_name] = []
            subset_dft[d3_name] = []
            wtmad_1_dft[d3_name] = []
            wtmad_2_dft[d3_name] = []
        processed = []

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_ele")] = (
                np.mean(data_subset[name_subset]["error_scf_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_ele")] = (
                np.mean(data_subset[name_subset]["error_dft_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_dip")] = (
                np.mean(data_subset[name_subset]["error_scf_dip"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_dip")] = (
                np.mean(data_subset[name_subset]["error_dft_dip"])
            )

            if len(data_subset[name_subset]["ai"]) == 0:
                for col_name in [
                    "AI",
                    "DFT",
                    "AI_D3BJ",
                    "DFT_D3BJ",
                    "AI_D3ZERO",
                    "DFT_D3ZERO",
                ]:
                    df_summary_subset.loc[i_subset, (data_path_name, col_name)] = 0
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                for d3_name in ["", "_d3bj", "_d3zero"]:
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"AI{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"ai{d3_name}"])
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"DFT{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"dft{d3_name}"])
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    "DONE"
                    if (
                        (
                            len(data_subset[name_subset]["ai"])
                            == len(data_subset[name_subset]["name"])
                        )
                        and (len(data_subset[name_subset]["ai"]) != 0)
                    )
                    else f"{len(data_subset[name_subset]['ai'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                for d3_name in ["", "_d3bj", "_d3zero"]:
                    subset_ai[d3_name] = np.append(
                        subset_ai[d3_name], data_subset[name_subset][f"ai{d3_name}"]
                    )
                    subset_dft[d3_name] = np.append(
                        subset_dft[d3_name], data_subset[name_subset][f"dft{d3_name}"]
                    )
                    wtmad_1_ai[d3_name] = np.append(
                        wtmad_1_ai[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"ai{d3_name}"]),
                    )
                    wtmad_1_dft[d3_name] = np.append(
                        wtmad_1_dft[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"dft{d3_name}"]),
                    )
                    wtmad_2_ai[d3_name] = np.append(
                        wtmad_2_ai[d3_name],
                        data_subset[name_subset][f"ai{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                    wtmad_2_dft[d3_name] = np.append(
                        wtmad_2_dft[d3_name],
                        data_subset[name_subset][f"dft{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["ai"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)
        for d3_name in ["", "_d3bj", "_d3zero"]:
            mean_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(subset_ai[d3_name])
            )
            mean_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(subset_dft[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(wtmad_1_ai[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(wtmad_1_dft[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.sum(wtmad_2_ai[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.sum(wtmad_2_dft[d3_name])
            )
        mean_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = np.mean(mean_absolute_deviation_list) / len(
        mean_absolute_deviation_list
    )
    print(
        f"Mean absolute deviation for {data_path_name}: {mean_absolute_deviation:.4f} kcal/mol"
    )
    for name_set in full_subset_dict.keys():
        for d3_name in ["", "_d3bj", "_d3zero"]:
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[
                    name_set, (data_path_name, f"DFT{d3_name.upper()}")
                ]
            )

# print("Summary")
# display(df_summary_subset_ele)
print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# # save summary to csv with date
# df_summary_subset_ele.to_csv(f"../validate/df_summary_subset_ele_{date}.csv")
# df_summary_subset.to_csv(f"../validate/summary_subset_{date}.csv")
# mean_subset.to_csv(f"../validate/mean_subset_{date}.csv")
# wtmad_1_subset.to_csv(f"../validate/wtmad_1_subset_{date}.csv")
# wtmad_2_subset.to_csv(f"../validate/wtmad_2_subset_{date}.csv")
# # save summary to excel with date
# df_summary_subset_ele.to_excel(f"../validate/df_summary_subset_ele_{date}.xlsx")
# df_summary_subset.to_excel(f"../validate/summary_subset_{date}.xlsx")
# mean_subset.to_excel(f"../validate/mean_subset_{date}.xlsx")
# wtmad_1_subset.to_excel(f"../validate/wtmad_1_subset_{date}.xlsx")
# wtmad_2_subset.to_excel(f"../validate/wtmad_2_subset_{date}.xlsx")

cc-pVDZ
Top 1 AI: 34.233055776202036 kcal/mol, 11 in 4192002_W4_11
  -1 * W4_11-b2h6  -1 * 10.695407580642495  2 * W4_11-b  2 * -0.022044754356102203  6 * W4_11-h  6 * -3.915593114474518
Top 2 AI: 33.56038100627484 kcal/mol, 131 in 4192002_W4_11
  -1 * W4_11-oclo  -1 * -33.21436206129147  2 * W4_11-o  2 * -0.048638512183970306  1 * W4_11-cl  1 * 0.4432959693367593
Top 3 AI: 32.95333680184558 kcal/mol, 107 in 4192002_W4_11
  -1 * W4_11-so3  -1 * -34.15919210616266  1 * W4_11-s  1 * -1.0599397677578963  3 * W4_11-o  3 * -0.048638512183970306
Top 4 AI: 30.548958831859636 kcal/mol, 136 in 4192002_W4_11
  -1 * W4_11-foof  -1 * -29.62172775104409  2 * W4_11-f  2 * 0.512254052591743  2 * W4_11-o  2 * -0.048638512183970306
Top 5 AI: 28.770819377125008 kcal/mol, 133 in 4192002_W4_11
  -1 * W4_11-b2  -1 * 28.726729868412804  2 * W4_11-b  2 * -0.022044754356102203
Top 1 DFT: 64.94188301078975 kcal/mol
Top 2 DFT: 64.82774996492662 kcal/mol
Top 3 DFT: 58.786665937819635 kcal/mol
Top 4 DFT: 55.62329

data_path   4192002                                                       \
Disp type        AI        DFT   AI_D3BJ   DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1       7.397052  13.710737  7.705655  14.259581  7.454815  13.713677   
sub2       7.949106   6.617683  11.14734   9.133989  8.647369   6.763003   
sub3       5.404118   6.261376  6.312986   7.355579  5.999373   6.934991   
sub4       3.857649   3.272197  4.840664    5.03074  4.815184   4.988172   
sub5       1.653929   1.320811  1.187383   0.927486  1.103389   0.872124   

data_path             1513512                        ...                       \
Disp type Processed        AI        DFT    AI_D3BJ  ... AI_D3ZERO DFT_D3ZERO   
sub1           DONE  6.282354  13.717062   6.868466  ...  6.365582  13.720054   
sub2           DONE  7.536765   6.612828  11.817093  ...  8.930029   6.760778   
sub3           DONE  4.620451    6.26131   5.733397  ...  5.317836   6.935442   
sub4           DONE   2.38953   3.272965   3.818374  ...  3.790518   4.986451   
sub5           DONE  1.250183   1.322771   0.717525  ...  0.665034   0.872943   

data_path             4190857                                            \
Disp type Processed        AI        DFT   AI_D3BJ   DFT_D3BJ AI_D3ZERO   
sub1           DONE  5.878346  13.710809  6.034077  14.259652  5.683363   
sub2           DONE  8.342449   6.615367  7.457728   9.134314  6.206933   
sub3           DONE  4.084904   6.261376  4.909066   7.355579  4.528621   
sub4           DONE  1.841768   3.272197  2.317571    5.03074  2.296708   
sub5           DONE  1.494092   1.320947  0.552164   0.927497  0.591902   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1       13.713748      DONE  
sub2        6.763238      DONE  
sub3        6.934991      DONE  
sub4        4.988172      DONE  
sub5        0.872207      DONE  

[5 rows x 21 columns]

wtmad_1


data_path    4192002                                                        \
Disp type         AI        DFT    AI_D3BJ  DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
sub1        8.026571   7.420931   8.407886  7.414587   7.954307   7.102103   
sub2       14.039316  12.535436  13.432814  9.971579  13.081751  10.010194   
sub3        6.665281   6.984991    7.67756  8.150912   7.244838   7.521852   
sub4       13.029252  11.029773  11.229165  14.87552  10.254653  13.798614   
sub5       14.524759  11.569831  10.209795  8.089137   9.674365   7.520409   

data_path              1513512                        ...             \
Disp type Processed         AI        DFT    AI_D3BJ  ...  AI_D3ZERO   
sub1           DONE   5.917804   7.427913   6.173631  ...   5.786012   
sub2           DONE   9.677109  12.530346   7.731942  ...   7.407743   
sub3           DONE   5.362819   6.982875   6.557344  ...   5.935941   
sub4           DONE  10.472747  11.046974  12.507724  ...  11.478243   
sub5           DONE  10.880025  11.599184   6.460544  ...   5.881129   

data_path                         4190857                                 \
Disp type DFT_D3ZERO Processed         AI        DFT   AI_D3BJ  DFT_D3BJ   
sub1        7.109349      DONE   6.332236    7.42093  5.740168  7.414586   
sub2       10.004803      DONE  12.258863  12.533853  9.365034  9.970671   
sub3        7.519994      DONE   4.930522   6.984991  5.759201  8.150912   
sub4       13.779798      DONE  11.161453  11.029773  9.252237  14.87552   
sub5        7.508553      DONE  13.595049    11.5704  4.671735  8.089186   

data_path                                 
Disp type AI_D3ZERO DFT_D3ZERO Processed  
sub1       5.713986   7.102103      DONE  
sub2       9.645465  10.008789      DONE  
sub3        5.21121   7.521852      DONE  
sub4       8.236251  13.798614      DONE  
sub5       5.203513   7.520755      DONE  

[5 rows x 21 columns]

wtmad_2


data_path   4192002                                                     \
Disp type        AI       DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1       3.730481  4.040696  3.896445  4.111446  3.771451   3.989769   
sub2       4.652822  3.640223  3.680039  2.379436  3.810301   2.550573   
sub3       2.204726  2.611064  2.567649  3.040013  2.451917   2.873647   
sub4       5.074244  4.159493  5.673725  6.856534  5.451359    6.62018   
sub5       6.127489   4.65963  4.404013  3.507041  4.142926   3.193414   

data_path             1513512                      ...                       \
Disp type Processed        AI       DFT   AI_D3BJ  ... AI_D3ZERO DFT_D3ZERO   
sub1           DONE  2.668675  4.045306  2.808633  ...  2.675278   3.994343   
sub2           DONE  3.206768  3.637148  2.047904  ...  2.176739   2.547486   
sub3           DONE  1.989474  2.610843   2.42163  ...  2.256968   2.873547   
sub4           DONE  4.130245   4.16366  6.081695  ...  5.851345   6.614613   
sub5           DONE  4.368745  4.670544  2.805652  ...  2.496356   3.192324   

data_path             4190857                                          \
Disp type Processed        AI       DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO   
sub1           DONE  2.896385  4.040699  2.747327  4.111449  2.718933   
sub2           DONE  4.263586  3.638673  2.879648   2.37847  3.129725   
sub3           DONE  1.749779  2.611064  2.078028  3.040013  1.921636   
sub4           DONE  4.368593  4.159493  4.530199  6.856534   4.30781   
sub5           DONE  4.982498  4.660409  2.000387  3.507108   2.04441   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1        3.989772      DONE  
sub2        2.549091      DONE  
sub3        2.873647      DONE  
sub4         6.62018      DONE  
sub5        3.193887      DONE  

[5 rows x 21 columns]

Summary of Subset
MAE


data_path    4192002                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
W4_11       10.66333  29.506381  10.833318  31.211271  10.623928  29.860573   
G21EA       1.368797   9.754713   1.365484   9.751936    1.37575   9.757737   
G21IP       1.887165   8.953509    1.89351   8.946874   1.890371   8.952075   
DIPCS10      2.76982  12.310852   2.750274  12.334099   2.747213  12.264655   
PA26        2.130008   2.200766   2.232988   1.900152   2.112959    2.00824   
SIE4x4     19.372674  21.908516  19.724808  22.337814   19.80166  22.339928   
ALKBDE10   14.584296  18.125246  15.272899  18.838948  14.594568  18.135588   
YBDE18       7.88327   8.145393   6.566602   7.265227   6.804269   7.472761   
AL2X6       2.528306   5.659145    4.67199   1.332003   2.693071   1.829895   
HEAVYSB11   5.352169   5.391476    8.05375   8.010474    6.02743   5.984154   
NBPRC       5.194536    2.23255   2.793279   2.544088   2.964373   2.274234   
ALK8        6.517239   4.400063   3.987358   3.174803   3.855053   2.559066   
RC21        5.089529   4.820926   6.467859   6.671006   5.915758     6.0297   
G2RC        7.969402   5.917203   8.800259   6.933515   8.369664   6.417298   
BH76RC      4.928623   3.484561   5.072159   3.539828    4.95332   3.536982   
FH51        5.458383   3.703657   6.406194    3.39478   6.225212   3.256908   
TAUT15      3.766778    2.16453   3.818437   2.175485   3.743546   2.145937   
DC13       18.818245  13.090861  22.203919  12.474047  20.424495  11.475687   
MB16_43    16.865727  15.459572  36.669437  36.862016  23.734956  23.162234   
DARC        8.553153   10.75415  16.274083    3.43411  13.729333    5.57797   
RSE43       3.379095     3.1576   3.152551   2.916276   2.998074    2.74911   
BSR36      12.979174   8.435801   5.625862    1.23793   7.657312   3.113938   
CDIE20      1.185442   1.599507   0.842494   1.336544   0.850532   1.347484   
ISO34       4.039369   2.001743   3.886036   1.577125   3.840333   1.647042   
PArel       2.537158   1.743933   2.462631   1.733749   2.548572   1.645025   
BH76        6.017022   9.107946   6.562533     9.8892   6.442369   9.676053   
BHPERI      6.450168   3.268992  11.044106   7.804724   9.639775     6.4139   
BHDIV10     5.348389   6.277706    6.20967   7.448955   5.445819   6.535109   
INV24       4.730775   2.697333   3.681237   2.346995   4.009396   2.228547   
BHROT27     0.852886   0.853379   0.869294   0.858852   0.864225    0.78352   
PX13        9.424224  11.042023  10.208212   11.82601   9.602058  11.219857   
WCPT18      6.157524   7.967151   7.344225   9.151986   6.932203   8.744304   
RG18        0.291302   0.230018   0.622562   0.655714   0.676401   0.709552   
ADIM6       3.770998   3.059029   1.378287   2.090256   1.678855   2.390824   
S22         2.960235   2.291252   1.468083   2.402928   1.371008   2.291758   
S66         2.430561   1.894709   1.238997   2.058561   1.284896   2.101976   
WATER27    17.253654  16.299413   26.44529  25.491048  27.357125  26.402884   
CARBHB12    1.021393   1.632408   1.984726   2.973103   1.853399   2.841777   
PNICO23     0.904183   0.776429   2.195519   2.604046    1.31829   1.726817   
HAL59       1.944712   1.649786   2.274223   2.629912   1.904915   2.247276   
AHB21        3.63609   2.453821   4.619029   3.464384   4.449164   3.294519   
CHB6        4.487423    1.80532   4.134877   3.038126   4.233166   2.309161   
IL16        2.390561   1.021067   4.984156   4.339132   5.054987   4.409963   
IDISP      10.362743  13.125475   6.074243   4.423061   5.139912   5.245746   
ICONF       0.628185   0.413804   0.521771   0.464652   0.602117   0.493337   
ACONF       0.548606   0.546621   0.459131   0.430751   0.334257   0.274429   
Amino20x4   1.032519   0.654639   0.901487   0.644234   0.836632   0.574733   
PCONF21     3.473282   3.214782   1.152402   1.178453   1.218123   1.099663   
MCONF        2.70061    1.95602   0.928197   0.741